# 05 · Logistic regression with DeepNetts

The tooling stream stopped at a train/test split. This is the other half: a real
model on real data.

We use **DeepNetts**, a pure-Java deep-learning library. A logistic regression is
just a single-layer neural network with a sigmoid output, so it is the natural
first model — and it reuses the same *load → scale → split → train → evaluate*
loop as the bigger networks that follow.


In [0]:
addDependency("com.deepnetts:deepnetts-core:4.0.4");

import deepnetts.net.FeedForwardNetwork;
import deepnetts.net.layers.activation.ActivationType;
import deepnetts.net.loss.LossType;
import deepnetts.data.DataSets;
import deepnetts.data.MLDataItem;
import deepnetts.data.TabularDataSet;
import deepnetts.data.TrainTestSplit;
import deepnetts.data.norm.MaxScaler;
import javax.visrec.ml.eval.EvaluationMetrics;
import java.nio.file.Files;
import java.nio.file.Path;

// The notebook lives in notebooks/exercises/, so walk up to find the repo's
// data/ directory - the same trick the other notebooks use.
Path dataset() {
    for (Path p = cwd.toAbsolutePath(); p != null; p = p.getParent()) {
        Path candidate = p.resolve("data/credit_card_fraud_imbalanced.csv");
        if (Files.isRegularFile(candidate)) return candidate;
    }
    throw new IllegalStateException("data/credit_card_fraud_imbalanced.csv not found above " + cwd);
}


## 1 · Load the transactions

`DataSets.readCsv` reads rows into a `TabularDataSet`: `7` feature columns and
`1` label column (`fraud`), with a header row. One million rows.


In [0]:
TabularDataSet<MLDataItem> dataSet = DataSets.readCsv(dataset().toString(), 7, 1, true);
println("loaded %d rows", dataSet.size());


## 2 · Scale

Features live on wildly different scales (a distance vs. a 0/1 flag). Scale every
column into `[0, 1]` so gradient descent behaves.


In [0]:
MaxScaler scaler = DataSets.scaleToMax(dataSet);
println("scaled to [0, 1]");


## 3 · Split before you train

Never evaluate on the rows you trained on — that only measures memorisation. Hold
back a fifth of the data for the test set.


In [0]:
TrainTestSplit split = DataSets.trainTestSplit(dataSet, 0.8);
var trainSet = split.getTrainingSet();
var testSet = split.getTestSet();
println("train %d   test %d", trainSet.size(), testSet.size());


## 4 · The model

A logistic regression: one input layer, one sigmoid output, cross-entropy loss.


In [0]:
FeedForwardNetwork logisticRegression = FeedForwardNetwork.builder()
        .addInputLayer(7)
        .addOutputLayer(1, ActivationType.SIGMOID)
        .lossFunction(LossType.CROSS_ENTROPY)
        .build();

logisticRegression.getTrainer()
        .setLearningRate(0.01f)
        .setStopError(0.13f)
        .setStopEpochs(100)
        .setShuffle(true);


In [0]:
// DeepNetts logs through java.util.logging; route it into the notebook.
class InternalLogHandler extends java.util.logging.Handler {
    public void close() {}
    public void flush() {}
    public void publish(java.util.logging.LogRecord logRecord) {
        println(logRecord.getMessage());
    }
}

deepnetts.core.DeepNetts.LOGGER.addHandler(new InternalLogHandler());


## 5 · Train


In [0]:
logisticRegression.train(trainSet);


## 6 · Evaluate on the held-out set


In [0]:
EvaluationMetrics evaluation = logisticRegression.test(testSet);
println(evaluation);
